<a href="https://colab.research.google.com/github/cerenckn/Image_Processing/blob/master/droneyes_model_e%C4%9Fitimi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 42.9/112.6 GB disk)


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!unzip -q "/content/drive/MyDrive/archive.zip" -d "/content/dataset"

**model eğitimi**

In [ ]:
!yolo task=detect mode=train model=yolov8n.pt data="/content/dataset/data.yaml" epochs=30 imgsz=640

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, pe

**MODEL BAŞARISI**

In [6]:
from ultralytics import YOLO

print("🚀 DOĞRULAMA (VALIDATION) BAŞLIYOR...")

# 1. Elindeki şampiyon modeli yükle (Dosya yolunu kendine göre ayarla)
model = YOLO("/content/best.pt")

# 2. Doğrulama işlemini başlat
# ÖNEMLİ: Veri setinin 'data.yaml' yolunu buraya doğru girmelisin
metrics = model.val(data="/content/dataset/data.yaml")

# 3. Sonuçları jilet gibi ekrana yazdırıyoruz
print("\n" + "="*40)
print("🎯 MODELİN KARNE SONUÇLARI 🎯")
print("="*40)
print(f"📌 Ortalama Doğruluk (mAP50) : %{metrics.box.map50 * 100:.2f}")
print(f"📌 Genel Doğruluk (mAP50-95): %{metrics.box.map * 100:.2f}")
print(f"📌 Hassasiyet (Precision)   : %{metrics.box.mp * 100:.2f}")
print(f"📌 Duyarlılık (Recall)      : %{metrics.box.mr * 100:.2f}")
print("="*40)

🚀 DOĞRULAMA (VALIDATION) BAŞLIYOR... 4 SAAT DEĞİL, 2 DAKİKA SÜRECEK!
Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1120.9±348.3 MB/s, size: 42.8 KB)
val: Scanning /content/dataset/valid/labels... 2523 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2523/2523 1.1Kit/s 2.3s
val: /content/dataset/valid/images/FLIR_05639_jpeg.rf.1fab114f3855b187e489f54b076bb16f.jpg: 1 duplicate labels removed
val: New cache created: /content/dataset/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 158/158 6.4it/s 24.5s
                   all       2523      23369      0.868      0.778      0.857      0.496
                   car       2275      13718      0.872      0.795      0.891      0.576
                   dog        134        150      0.882      0.746   

****

In [13]:
import cv2
from ultralytics import YOLO

# 1. Eğittiğimiz beyni (modeli) yüklüyoruz
model_path = "/content/best.pt"
model = YOLO(model_path)

# 2. İşleyeceğimiz kısa videomuz (Sahne 1'i seçtik)
input_video_path = "/content/kesik_video2.mp4"
output_video_path = "/content/hedef_kilitlendi_super_net.mp4"

# Videoyu okuma ve yazma ayarları
cap = cv2.VideoCapture(input_video_path)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

# --- EFSANE DOKUNUŞ 1: CLAHE FİLTRESİ ---
# Kontrastı artırıp pikselleri netleştirecek yapı
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))

# --- EFSANE DOKUNUŞ 2: SINIFLARA GÖRE RENKLER ---
# (BGR formatında) 0 genelde insan, 1 veya 2 araçtır.
# FLIR veri setindeki sıraya göre bunu model kendi isimlendirecek.
renk_sozlugu = {
    0: (0, 0, 255),    # Sınıf 0 (İnsan/Piyade) -> Neon Kırmızı
    1: (0, 255, 255),  # Sınıf 1 (Araç vb) -> Sarı
    2: (255, 100, 0),  # Sınıf 2 -> Mavi/Turuncu tonları
}

print("CLAHE Görüntü Netleştirici devrede... Hedefler farklı renklerde aranıyor, lütfen bekle bebeğim.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Önce görüntüyü griye çevirip CLAHE ile pikselleri netleştiriyoruz
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    enhanced_gray = clahe.apply(gray_frame)
    # YOLO 3 kanal beklediği için tekrar renkli formata büründürüyoruz
    enhanced_frame = cv2.cvtColor(enhanced_gray, cv2.COLOR_GRAY2BGR)

    # Modeli Orijinal değil, NETLEŞTİRİLMİŞ görüntünün üzerine salıyoruz! (Çok daha iyi yakalar)
    results = model(enhanced_frame, conf=0.4)

    hedef_var_mi = False

    for r in results:
        boxes = r.boxes
        for box in boxes:
            hedef_var_mi = True

            # Koordinatları ve o nesnenin sınıf (class) numarasını alıyoruz
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            cls_id = int(box.cls[0])
            sinif_ismi = model.names[cls_id].upper() # Örneğin "PERSON", "CAR"

            # O sınıfa ait rengi sözlükten çek (Sözlükte yoksa default Yeşil kullan)
            renk = renk_sozlugu.get(cls_id, (0, 255, 0))

            # Ekrana kutuyu çiziyoruz
            cv2.rectangle(enhanced_frame, (x1, y1), (x2, y2), renk, 2)

            # Hedefin tam ortasına o renkli askeri artı (crosshair) işareti
            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)
            cv2.drawMarker(enhanced_frame, (center_x, center_y), renk,
                           markerType=cv2.MARKER_CROSS, markerSize=15, thickness=2, line_type=cv2.LINE_AA)

            # Kutunun hemen üstüne bulduğu şeyin ne olduğunu da o renkte yazdıralım (Çok profesyonel durur)
            cv2.putText(enhanced_frame, sinif_ismi, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, renk, 2)

    # Eğer ekranda hedefler varsa kocaman kırmızı uyarımızı basıyoruz
    if hedef_var_mi:
        cv2.putText(enhanced_frame, "HEDEF KILITLENDI - ATISA HAZIR", (50, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3, cv2.LINE_AA)

    # İşlenmiş net kareyi yeni videoya ekle
    out.write(enhanced_frame)

cap.release()
out.release()
print("Görev Tamamlandı! Havalı ve net sonuç videon sol menüye 'hedef_kilitlendi_super_net.mp4' olarak eklendi.")

CLAHE Görüntü Netleştirici devrede... Hedefler farklı renklerde aranıyor, lütfen bekle bebeğim.

0: 384x640 1 person, 9.4ms
Speed: 2.2ms preprocess, 9.4ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 6.2ms
Speed: 3.2ms preprocess, 6.2ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 6.3ms
Speed: 3.7ms preprocess, 6.3ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 7.4ms
Speed: 2.8ms preprocess, 7.4ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 6.8ms
Speed: 2.9ms preprocess, 6.8ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 7.5ms
Speed: 2.6ms preprocess, 7.5ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 7.1ms
Speed: 2.4ms preprocess, 7.1ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no

In [14]:
import cv2
import numpy as np
from ultralytics import YOLO

# 1. Modeli ve videoyu yüklüyoruz
model_path = "/content/best.pt"
model = YOLO(model_path)

input_video_path = "/content/kesik_video.mp4"
output_video_path = "/content/hedef_kilitlendi_agir_cekim.mp4"

cap = cv2.VideoCapture(input_video_path)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = int(cap.get(cv2.CAP_PROP_FPS))

# --- EFSANE DOKUNUŞ 1: AĞIR ÇEKİM (SLOW-MOTION) ---
# FPS'i yarıya bölüyoruz, video 2 kat yavaşlayıp sinematik olacak
slow_fps = original_fps // 2
if slow_fps < 10: slow_fps = 10

out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), slow_fps, (width, height))

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

# --- EFSANE DOKUNUŞ 2: KESKİNLEŞTİRME MATRİSİ ---
# Piksellerin sınırlarını belirginleştiren matematiksel matris
sharpen_kernel = np.array([[-1,-1,-1],
                           [-1, 9,-1],
                           [-1,-1,-1]])

renk_sozlugu = {
    0: (0, 0, 255),    # İnsan -> Kırmızı
    1: (0, 255, 255)   # Araba -> Sarı
}

print("Gelişmiş Radar devrede... Ağır çekim ve Keskinleştirme uygulanıyor bebeğim.")

hedef_yazi_sayaci = 0 # Yazının ekranda asılı kalması için sayaç

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Önce görüntüyü jilet gibi keskinleştiriyoruz
    sharpened_frame = cv2.filter2D(frame, -1, sharpen_kernel)

    # Sonra CLAHE uyguluyoruz
    gray_frame = cv2.cvtColor(sharpened_frame, cv2.COLOR_BGR2GRAY)
    enhanced_gray = clahe.apply(gray_frame)
    enhanced_frame = cv2.cvtColor(enhanced_gray, cv2.COLOR_GRAY2BGR)

    # --- EFSANE DOKUNUŞ 3: GÜVEN SKORUNU ARTIR (0.4 yerine 0.6) ---
    results = model(enhanced_frame, conf=0.6)

    hedef_bulundu_bu_karede = False

    for r in results:
        boxes = r.boxes
        for box in boxes:
            cls_id = int(box.cls[0])
            sinif_ismi = model.names[cls_id].upper()

            # --- EFSANE DOKUNUŞ 4: KATI FİLTRELEME ---
            # Eğer sistem bulduğu şeye PERSON veya CAR demiyorsa direkt yok say! (DOG vs elenir)
            if sinif_ismi not in ["PERSON", "CAR"]:
                continue

            hedef_bulundu_bu_karede = True

            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

            renk = renk_sozlugu.get(cls_id, (0, 255, 0))

            # Okunabilirlik için çizgileri kalınlaştırdık (thickness=3)
            cv2.rectangle(enhanced_frame, (x1, y1), (x2, y2), renk, 3)

            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)
            cv2.drawMarker(enhanced_frame, (center_x, center_y), renk,
                           markerType=cv2.MARKER_CROSS, markerSize=30, thickness=3, line_type=cv2.LINE_AA)

            cv2.putText(enhanced_frame, sinif_ismi, (x1, y1 - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.9, renk, 3)

    # --- EFSANE DOKUNUŞ 5: YAZIYI EKRANDA TUTMA ---
    if hedef_bulundu_bu_karede:
        hedef_yazi_sayaci = 15 # Hedef bulursa yazıyı 15 kare (yaklaşık yarım saniye) ekranda tut

    if hedef_yazi_sayaci > 0:
        cv2.putText(enhanced_frame, "HEDEF KILITLENDI - ATISA HAZIR", (30, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 0, 255), 4, cv2.LINE_AA)
        hedef_yazi_sayaci -= 1

    # Ağır çekimde kaydediyoruz
    out.write(enhanced_frame)

cap.release()
out.release()
print("Görev Tamamlandı! Ağır çekim ve hatasız video sol tarafa 'hedef_kilitlendi_agir_cekim.mp4' olarak eklendi.")

Gelişmiş Radar devrede... Ağır çekim ve Keskinleştirme uygulanıyor bebeğim.

0: 480x640 (no detections), 9.5ms
Speed: 2.5ms preprocess, 9.5ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.8ms
Speed: 2.4ms preprocess, 9.8ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 10.5ms
Speed: 3.0ms preprocess, 10.5ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 13.0ms
Speed: 2.9ms preprocess, 13.0ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 14.1ms
Speed: 3.5ms preprocess, 14.1ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 12.0ms
Speed: 2.8ms preprocess, 12.0ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 8.1ms
Speed: 3.0ms preprocess, 8.1ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

 **VİDEO İŞLEME KISMI**

video-1

In [8]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque
import time
import datetime
import os

print("🛡️ AEGIS v2.0 (output_v2) ANA MOTOR BAŞLATILIYOR...")

# ══════════════════════════════════════════════════════════════════════════════
# 1. AYARLAR VE YAPILANDIRMA (SİSTEM PARAMETRELERİ)
# ══════════════════════════════════════════════════════════════════════════════
MODEL_PATH   = "/content/best.pt"
VIDEO_PATH   = "/content/kesik_video.mp4"
OUTPUT_PATH  = "/content/output_v2.mp4"

# Modelin hassasiyet ayarı. 0.45 altındaki nesneleri (taş, gölge) görmezden gelir.
CONF         = 0.45

# Sınıf Filtresi: Sadece İnsan(0), Araba(2) ve Kamyon(7) tespit edilsin. Köpek/Kuş elensin.
ALLOWED_CLASSES = [0, 2, 7]

# Minimum Bounding Box (Kutu) Boyutu. Çok uzaktaki minik parazitleri engeller.
MIN_BOX_W    = 30
MIN_BOX_H    = 40

# Ayak izi kuyruğunun uzunluğu (Kaç karelik geçmişi çizecek)
TRAIL_LEN    = 55

# Sanal Güvenlik Bölgesi Koordinatları (Ekrana oranla: Sol %12, Üst %10, Sağ %88, Alt %90)
ZONE         = (0.12, 0.10, 0.88, 0.90)
BREACH_HOLD  = 80  # Sınır ihlali alarmı tetiklendiğinde ekranda kaç kare (frame) kalacak?

# Hız hesaplaması için eşik değerler (Piksel / Frame cinsinden)
SPEED_MEDIUM = 8
SPEED_HIGH   = 20

# ══════════════════════════════════════════════════════════════════════════════
# 2. RENK PALETİ (ASKERİ HUD TASARIMI İÇİN BGR FORMATINDA)
# ══════════════════════════════════════════════════════════════════════════════
C_GREEN   = (0,  255, 100) # Güvenli nesneler ve HUD çizgileri
C_BLUE    = (255, 160,  50) # Araçlar
C_RED     = (0,    0, 230) # İhlal ve Yüksek Tehdit
C_CYAN    = (255, 255,   0) # Kilitlenmiş hedef
C_ORANGE  = (0,  140, 255) # Orta Tehdit
C_DIM     = (0,   70,   0) # HUD arka plan detayları
C_FOOT    = (0,  200, 255) # Ayak izi rengi (Sarımsı)

# ══════════════════════════════════════════════════════════════════════════════
# 3. YARDIMCI FONKSİYONLAR (MATEMATİK VE ÇİZİM ALGORİTMALARI)
# ══════════════════════════════════════════════════════════════════════════════

def get_zone_px(W, H):
    """Oransal olarak verilen ZONE değerlerini, videonun gerçek piksel çözünürlüğüne çevirir."""
    return (int(ZONE[0]*W), int(ZONE[1]*H), int(ZONE[2]*W), int(ZONE[3]*H))

def bbox_breaches_zone(x1, y1, x2, y2, zx1, zy1, zx2, zy2):
    """
    Sınır İhlali Mantığı: Sadece merkeze değil, kutunun 4 köşesinden herhangi biri
    çizdiğimiz alanın DIŞINA çıkarsa veya ÇARPARSA ihlal (True) sayılır.
    """
    points = [(x1, y1), (x2, y1), (x1, y2), (x2, y2), ((x1+x2)//2, (y1+y2)//2)]
    for (px, py) in points:
        if px < zx1 or px > zx2 or py < zy1 or py > zy2:
            return True
    return False

def compute_speed(hist):
    """
    Hedefin hızını hesaplar: Son 8 karedeki merkez noktaları arasındaki
    mesafelerin ortalamasını (Piksel/Frame) alarak anlık hızı bulur.
    """
    if len(hist) < 3: return 0.0
    dists = []
    for i in range(1, len(hist)):
        dx = hist[i][0] - hist[i-1][0]
        dy = hist[i][1] - hist[i-1][1]
        dists.append((dx**2 + dy**2) ** 0.5) # Pisagor teoremi ile mesafe
    return sum(dists) / len(dists)

def distance_estimate(bbox_h, class_id=0):
    """
    Mesafe Tahmini: Kameranın odak uzaklığı (Focal=700) ve nesnenin gerçek dünyadaki
    ortalama yüksekliği kullanılarak uzaklık metre cinsinden hesaplanır.
    Formül: Mesafe = (Gerçek Boy * Odak) / Piksel Boyu
    """
    if bbox_h < 5: return None
    real_heights = {0: 1.75, 2: 1.50, 7: 2.50} # İnsan 1.75m, Araba 1.5m, Kamyon 2.5m
    real_h = real_heights.get(class_id, 1.70)
    return round((real_h * 700) / bbox_h, 1)

def threat_level(dist, speed):
    """
    Yapay Zeka Karar Destek Sistemi:
    Hedef çok hızlıysa (SPEED_HIGH) veya çok yakınsa (<3 metre) KIRMIZI (HIGH) alarm verir.
    Aksi durumlarda hız ve mesafeye göre SARI (MEDIUM) veya YEŞİL (LOW) döner.
    """
    if dist is None and speed < SPEED_MEDIUM: return "LOW", C_GREEN
    dist_score  = 2 if (dist and dist < 3.0) else (1 if (dist and dist < 8.0) else 0)
    speed_score = 2 if speed >= SPEED_HIGH else (1 if speed >= SPEED_MEDIUM else 0)
    total = dist_score + speed_score

    if total >= 3: return "HIGH", C_RED
    if total >= 1: return "MEDIUM", C_ORANGE
    return "LOW", C_GREEN

def draw_foot_trail(frame, pts, color):
    """Hedefin arkasındaki ayak izlerini ve soluklaşan kuyruğu çizer."""
    if len(pts) < 2: return
    # Çizgi katmanı (soluklaşan)
    for i in range(1, len(pts)):
        alpha = i / len(pts)
        thickness = max(1, int(2 * alpha))
        c = tuple(int(ch * alpha * 0.6) for ch in color)
        cv2.line(frame, pts[i-1], pts[i], c, thickness, cv2.LINE_AA)
    # Adım noktaları
    for i, pt in enumerate(pts):
        if i % 4 == 0:
            alpha = (i + 1) / len(pts)
            c = tuple(int(ch * alpha) for ch in color)
            cv2.circle(frame, pt, max(2, int(4 * alpha)), c, -1, cv2.LINE_AA)

def draw_hud_frame(frame, W, H, fps, breach, n_targets):
    """Dron kamerasındaki o askeri köşe çizgilerini, yazıları ve ekran uyarılarını çizer."""
    blen, bthick, boff = 45, 2, 10
    corners = [(boff, boff), (W-boff-blen, boff), (boff, H-boff-blen), (W-boff-blen, H-boff-blen)]
    dirs    = [(1,1), (-1,1), (1,-1), (-1,-1)]
    for (cx, cy), (dx, dy) in zip(corners, dirs):
        cv2.line(frame, (cx, cy), (cx+dx*blen, cy), C_GREEN, bthick)
        cv2.line(frame, (cx, cy), (cx, cy+dy*blen), C_GREEN, bthick)

    cv2.putText(frame, "AEGIS v2.0 | BORDER WATCH", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.58, C_GREEN, 1, cv2.LINE_AA)
    cv2.putText(frame, f"FPS {fps:>4.1f}  TGT {n_targets:02d}", (W-230, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.58, C_GREEN, 1, cv2.LINE_AA)

    # Merkez artı işareti (Crosshair)
    mx, my = W//2, H//2
    cv2.line(frame, (mx-16, my), (mx+16, my), C_DIM, 1)
    cv2.line(frame, (mx, my-16), (mx, my+16), C_DIM, 1)
    cv2.circle(frame, (mx, my), 5, C_DIM, 1)

    # İhlal anında yanıp sönen dev kırmızı uyarı
    if breach > 0 and (breach % 16) < 8:
        cv2.rectangle(frame, (0, 0), (W-1, H-1), C_RED, 5)
        txt = "!! BORDER BREACH DETECTED !"
        tw = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 1.0, 2)[0][0]
        cv2.putText(frame, txt, (W//2 - tw//2 + 2, H//2 + 2), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,0), 3, cv2.LINE_AA)
        cv2.putText(frame, txt, (W//2 - tw//2, H//2), cv2.FONT_HERSHEY_SIMPLEX, 1.0, C_RED, 2, cv2.LINE_AA)

# ══════════════════════════════════════════════════════════════════════════════
# 4. SİSTEM BAŞLATMA VE ANA DÖNGÜ (VİDEO İŞLEME)
# ══════════════════════════════════════════════════════════════════════════════
if not os.path.exists(MODEL_PATH): raise FileNotFoundError("❌ 'best.pt' bulunamadı!")
if not os.path.exists(VIDEO_PATH): raise FileNotFoundError("❌ İşlenecek video bulunamadı!")

model = YOLO(MODEL_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)
W, H = int(cap.get(3)), int(cap.get(4))
FPS = cap.get(cv2.CAP_PROP_FPS) or 25
out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H))

center_history = defaultdict(lambda: deque(maxlen=8))
foot_trails = defaultdict(lambda: deque(maxlen=TRAIL_LEN))
breached_ids = set()
breach_timer, frame_idx, locked_id, prev_time, fps_display = 0, 0, None, time.time(), 0.0

zx1, zy1, zx2, zy2 = get_zone_px(W, H)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame_idx += 1

    # YOLO Takip Algoritması
    results = model.track(frame, conf=CONF, iou=0.50, classes=ALLOWED_CLASSES, persist=True, tracker="bytetrack.yaml", verbose=False)

    tracked_objs = []
    new_breach = False

    if results[0].boxes is not None and results[0].boxes.id is not None:
        boxes = results[0].boxes
        for i in range(len(boxes)):
            x1, y1, x2, y2 = map(int, boxes.xyxy[i].tolist())
            bw, bh = x2 - x1, y2 - y1

            # Gürültü Filtresi: Kutu çok küçükse atla
            if bw < MIN_BOX_W or bh < MIN_BOX_H: continue

            tid = int(boxes.id[i])
            cid = int(boxes.cls[i])
            cx, cy = (x1+x2)//2, (y1+y2)//2
            foot_pt = (cx, y2) # Ayak izi için alt orta nokta

            center_history[tid].append((cx, cy))
            foot_trails[tid].append(foot_pt)

            tracked_objs.append({
                "id": tid, "bbox": [x1, y1, x2, y2], "class_id": cid,
                "label": f"{model.names.get(cid, 'OBJ').upper()} ID{tid}",
                "bbox_h": bh
            })

            # İhlal Kontrolü
            if bbox_breaches_zone(x1, y1, x2, y2, zx1, zy1, zx2, zy2):
                breach_timer = BREACH_HOLD
                breached_ids.add(tid)
                new_breach = True

    if breach_timer > 0: breach_timer -= 1

    # FPS Hesabı
    now = time.time()
    fps_display = 0.88 * fps_display + 0.12 * (1.0 / max(now - prev_time, 1e-6))
    prev_time = now

    # En yakın hedefe kilitlenme (locked_id) hesaplaması
    if tracked_objs:
        locked_id = min(tracked_objs, key=lambda o: distance_estimate(o["bbox_h"], o["class_id"]) or 999)["id"]

    # --- ÇİZİM AŞAMASI ---
    draw_hud_frame(frame, W, H, fps_display, breach_timer, len(tracked_objs))

    # Önce ayak izlerini çiz (Kutuların altında kalsın)
    for obj in tracked_objs:
        tid, cid = obj["id"], obj["class_id"]
        draw_foot_trail(frame, list(foot_trails[tid]), C_FOOT if cid == 0 else C_BLUE)

    # Hedef Kutuları ve Metinleri
    for obj in tracked_objs:
        tid, cid = obj["id"], obj["class_id"]
        x1, y1, x2, y2 = obj["bbox"]

        dist = distance_estimate(obj["bbox_h"], cid)
        speed = compute_speed(list(center_history[tid]))
        is_locked = (tid == locked_id)
        is_breached = (tid in breached_ids)

        # Kutu Rengi
        color = C_RED if is_breached else (C_CYAN if is_locked else (C_BLUE if cid in (2, 7) else C_GREEN))

        # Ana kutu ve köşeler
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        for (px, py), (dx, dy) in [((x1,y1),(1,1)), ((x2,y1),(-1,1)), ((x1,y2),(1,-1)), ((x2,y2),(-1,-1))]:
            cv2.line(frame, (px, py), (px+dx*14, py), color, 2)
            cv2.line(frame, (px, py), (px, py+dy*14), color, 2)

        # Etiketler (İsim, Mesafe, Hız, Tehdit)
        lbl = obj["label"] + (f"  {dist}m" if dist else "") + (" [LOCKED]" if is_locked else "")
        (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.50, 1)
        lbl_y = max(y1, th + 14)
        cv2.rectangle(frame, (x1, lbl_y-th-10), (x1+tw+8, lbl_y), color, -1)
        cv2.putText(frame, lbl, (x1+4, lbl_y-4), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (0,0,0), 1, cv2.LINE_AA)

        lvl, lvl_color = threat_level(dist, speed)
        cv2.putText(frame, f"THREAT:{lvl}  SPD:{speed:.0f}px/f", (x1, y2+16), cv2.FONT_HERSHEY_SIMPLEX, 0.42, lvl_color, 1, cv2.LINE_AA)

    # İhlal anında güvenlik bölgesini kırmızı ve yarı saydam göster
    if breach_timer > 0:
        overlay = frame.copy()
        cv2.rectangle(overlay, (zx1, zy1), (zx2, zy2), C_RED, 2)
        cv2.putText(overlay, "SECURITY ZONE", (zx1+6, zy1+18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, C_RED, 1)
        cv2.addWeighted(overlay, 0.35, frame, 0.65, 0, frame)

    out.write(frame)

cap.release()
out.release()
print(f"✅ İŞLEM TAMAMLANDI! Çıktı: {OUTPUT_PATH}")
from google.colab import files
files.download(OUTPUT_PATH)

🛡️ AEGIS v2.0 (output_v2) ANA MOTOR BAŞLATILIYOR...
✅ İŞLEM TAMAMLANDI! Çıktı: /content/output_v2.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

video-2

In [9]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque
import time
import datetime
import os

print("🛡️ AEGIS v2.0 (output_v2) ANA MOTOR BAŞLATILIYOR...")

# ══════════════════════════════════════════════════════════════════════════════
# 1. AYARLAR VE YAPILANDIRMA (SİSTEM PARAMETRELERİ)
# ══════════════════════════════════════════════════════════════════════════════
MODEL_PATH   = "/content/best.pt"
VIDEO_PATH   = "/content/kesik_video2.mp4"
OUTPUT_PATH  = "/content/output_v1.mp4"

# Modelin hassasiyet ayarı. 0.45 altındaki nesneleri (taş, gölge) görmezden gelir.
CONF         = 0.45

# Sınıf Filtresi: Sadece İnsan(0), Araba(2) ve Kamyon(7) tespit edilsin. Köpek/Kuş elensin.
ALLOWED_CLASSES = [0, 2, 7]

# Minimum Bounding Box (Kutu) Boyutu. Çok uzaktaki minik parazitleri engeller.
MIN_BOX_W    = 30
MIN_BOX_H    = 40

# Ayak izi kuyruğunun uzunluğu (Kaç karelik geçmişi çizecek)
TRAIL_LEN    = 55

# Sanal Güvenlik Bölgesi Koordinatları (Ekrana oranla: Sol %12, Üst %10, Sağ %88, Alt %90)
ZONE         = (0.12, 0.10, 0.88, 0.90)
BREACH_HOLD  = 80  # Sınır ihlali alarmı tetiklendiğinde ekranda kaç kare (frame) kalacak?

# Hız hesaplaması için eşik değerler (Piksel / Frame cinsinden)
SPEED_MEDIUM = 8
SPEED_HIGH   = 20

# ══════════════════════════════════════════════════════════════════════════════
# 2. RENK PALETİ (ASKERİ HUD TASARIMI İÇİN BGR FORMATINDA)
# ══════════════════════════════════════════════════════════════════════════════
C_GREEN   = (0,  255, 100) # Güvenli nesneler ve HUD çizgileri
C_BLUE    = (255, 160,  50) # Araçlar
C_RED     = (0,    0, 230) # İhlal ve Yüksek Tehdit
C_CYAN    = (255, 255,   0) # Kilitlenmiş hedef
C_ORANGE  = (0,  140, 255) # Orta Tehdit
C_DIM     = (0,   70,   0) # HUD arka plan detayları
C_FOOT    = (0,  200, 255) # Ayak izi rengi (Sarımsı)

# ══════════════════════════════════════════════════════════════════════════════
# 3. YARDIMCI FONKSİYONLAR (MATEMATİK VE ÇİZİM ALGORİTMALARI)
# ══════════════════════════════════════════════════════════════════════════════

def get_zone_px(W, H):
    """Oransal olarak verilen ZONE değerlerini, videonun gerçek piksel çözünürlüğüne çevirir."""
    return (int(ZONE[0]*W), int(ZONE[1]*H), int(ZONE[2]*W), int(ZONE[3]*H))

def bbox_breaches_zone(x1, y1, x2, y2, zx1, zy1, zx2, zy2):
    """
    Sınır İhlali Mantığı: Sadece merkeze değil, kutunun 4 köşesinden herhangi biri
    çizdiğimiz alanın DIŞINA çıkarsa veya ÇARPARSA ihlal (True) sayılır.
    """
    points = [(x1, y1), (x2, y1), (x1, y2), (x2, y2), ((x1+x2)//2, (y1+y2)//2)]
    for (px, py) in points:
        if px < zx1 or px > zx2 or py < zy1 or py > zy2:
            return True
    return False

def compute_speed(hist):
    """
    Hedefin hızını hesaplar: Son 8 karedeki merkez noktaları arasındaki
    mesafelerin ortalamasını (Piksel/Frame) alarak anlık hızı bulur.
    """
    if len(hist) < 3: return 0.0
    dists = []
    for i in range(1, len(hist)):
        dx = hist[i][0] - hist[i-1][0]
        dy = hist[i][1] - hist[i-1][1]
        dists.append((dx**2 + dy**2) ** 0.5) # Pisagor teoremi ile mesafe
    return sum(dists) / len(dists)

def distance_estimate(bbox_h, class_id=0):
    """
    Mesafe Tahmini: Kameranın odak uzaklığı (Focal=700) ve nesnenin gerçek dünyadaki
    ortalama yüksekliği kullanılarak uzaklık metre cinsinden hesaplanır.
    Formül: Mesafe = (Gerçek Boy * Odak) / Piksel Boyu
    """
    if bbox_h < 5: return None
    real_heights = {0: 1.75, 2: 1.50, 7: 2.50} # İnsan 1.75m, Araba 1.5m, Kamyon 2.5m
    real_h = real_heights.get(class_id, 1.70)
    return round((real_h * 700) / bbox_h, 1)

def threat_level(dist, speed):
    """
    Yapay Zeka Karar Destek Sistemi:
    Hedef çok hızlıysa (SPEED_HIGH) veya çok yakınsa (<3 metre) KIRMIZI (HIGH) alarm verir.
    Aksi durumlarda hız ve mesafeye göre SARI (MEDIUM) veya YEŞİL (LOW) döner.
    """
    if dist is None and speed < SPEED_MEDIUM: return "LOW", C_GREEN
    dist_score  = 2 if (dist and dist < 3.0) else (1 if (dist and dist < 8.0) else 0)
    speed_score = 2 if speed >= SPEED_HIGH else (1 if speed >= SPEED_MEDIUM else 0)
    total = dist_score + speed_score

    if total >= 3: return "HIGH", C_RED
    if total >= 1: return "MEDIUM", C_ORANGE
    return "LOW", C_GREEN

def draw_foot_trail(frame, pts, color):
    """Hedefin arkasındaki ayak izlerini ve soluklaşan kuyruğu çizer."""
    if len(pts) < 2: return
    # Çizgi katmanı (soluklaşan)
    for i in range(1, len(pts)):
        alpha = i / len(pts)
        thickness = max(1, int(2 * alpha))
        c = tuple(int(ch * alpha * 0.6) for ch in color)
        cv2.line(frame, pts[i-1], pts[i], c, thickness, cv2.LINE_AA)
    # Adım noktaları
    for i, pt in enumerate(pts):
        if i % 4 == 0:
            alpha = (i + 1) / len(pts)
            c = tuple(int(ch * alpha) for ch in color)
            cv2.circle(frame, pt, max(2, int(4 * alpha)), c, -1, cv2.LINE_AA)

def draw_hud_frame(frame, W, H, fps, breach, n_targets):
    """Dron kamerasındaki o askeri köşe çizgilerini, yazıları ve ekran uyarılarını çizer."""
    blen, bthick, boff = 45, 2, 10
    corners = [(boff, boff), (W-boff-blen, boff), (boff, H-boff-blen), (W-boff-blen, H-boff-blen)]
    dirs    = [(1,1), (-1,1), (1,-1), (-1,-1)]
    for (cx, cy), (dx, dy) in zip(corners, dirs):
        cv2.line(frame, (cx, cy), (cx+dx*blen, cy), C_GREEN, bthick)
        cv2.line(frame, (cx, cy), (cx, cy+dy*blen), C_GREEN, bthick)

    cv2.putText(frame, "AEGIS v2.0 | BORDER WATCH", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.58, C_GREEN, 1, cv2.LINE_AA)
    cv2.putText(frame, f"FPS {fps:>4.1f}  TGT {n_targets:02d}", (W-230, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.58, C_GREEN, 1, cv2.LINE_AA)

    # Merkez artı işareti (Crosshair)
    mx, my = W//2, H//2
    cv2.line(frame, (mx-16, my), (mx+16, my), C_DIM, 1)
    cv2.line(frame, (mx, my-16), (mx, my+16), C_DIM, 1)
    cv2.circle(frame, (mx, my), 5, C_DIM, 1)

    # İhlal anında yanıp sönen dev kırmızı uyarı
    if breach > 0 and (breach % 16) < 8:
        cv2.rectangle(frame, (0, 0), (W-1, H-1), C_RED, 5)
        txt = "!! BORDER BREACH DETECTED !"
        tw = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 1.0, 2)[0][0]
        cv2.putText(frame, txt, (W//2 - tw//2 + 2, H//2 + 2), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,0), 3, cv2.LINE_AA)
        cv2.putText(frame, txt, (W//2 - tw//2, H//2), cv2.FONT_HERSHEY_SIMPLEX, 1.0, C_RED, 2, cv2.LINE_AA)

# ══════════════════════════════════════════════════════════════════════════════
# 4. SİSTEM BAŞLATMA VE ANA DÖNGÜ (VİDEO İŞLEME)
# ══════════════════════════════════════════════════════════════════════════════
if not os.path.exists(MODEL_PATH): raise FileNotFoundError("❌ 'best.pt' bulunamadı!")
if not os.path.exists(VIDEO_PATH): raise FileNotFoundError("❌ İşlenecek video bulunamadı!")

model = YOLO(MODEL_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)
W, H = int(cap.get(3)), int(cap.get(4))
FPS = cap.get(cv2.CAP_PROP_FPS) or 25
out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H))

center_history = defaultdict(lambda: deque(maxlen=8))
foot_trails = defaultdict(lambda: deque(maxlen=TRAIL_LEN))
breached_ids = set()
breach_timer, frame_idx, locked_id, prev_time, fps_display = 0, 0, None, time.time(), 0.0

zx1, zy1, zx2, zy2 = get_zone_px(W, H)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame_idx += 1

    # YOLO Takip Algoritması
    results = model.track(frame, conf=CONF, iou=0.50, classes=ALLOWED_CLASSES, persist=True, tracker="bytetrack.yaml", verbose=False)

    tracked_objs = []
    new_breach = False

    if results[0].boxes is not None and results[0].boxes.id is not None:
        boxes = results[0].boxes
        for i in range(len(boxes)):
            x1, y1, x2, y2 = map(int, boxes.xyxy[i].tolist())
            bw, bh = x2 - x1, y2 - y1

            # Gürültü Filtresi: Kutu çok küçükse atla
            if bw < MIN_BOX_W or bh < MIN_BOX_H: continue

            tid = int(boxes.id[i])
            cid = int(boxes.cls[i])
            cx, cy = (x1+x2)//2, (y1+y2)//2
            foot_pt = (cx, y2) # Ayak izi için alt orta nokta

            center_history[tid].append((cx, cy))
            foot_trails[tid].append(foot_pt)

            tracked_objs.append({
                "id": tid, "bbox": [x1, y1, x2, y2], "class_id": cid,
                "label": f"{model.names.get(cid, 'OBJ').upper()} ID{tid}",
                "bbox_h": bh
            })

            # İhlal Kontrolü
            if bbox_breaches_zone(x1, y1, x2, y2, zx1, zy1, zx2, zy2):
                breach_timer = BREACH_HOLD
                breached_ids.add(tid)
                new_breach = True

    if breach_timer > 0: breach_timer -= 1

    # FPS Hesabı
    now = time.time()
    fps_display = 0.88 * fps_display + 0.12 * (1.0 / max(now - prev_time, 1e-6))
    prev_time = now

    # En yakın hedefe kilitlenme (locked_id) hesaplaması
    if tracked_objs:
        locked_id = min(tracked_objs, key=lambda o: distance_estimate(o["bbox_h"], o["class_id"]) or 999)["id"]

    # --- ÇİZİM AŞAMASI ---
    draw_hud_frame(frame, W, H, fps_display, breach_timer, len(tracked_objs))

    # Önce ayak izlerini çiz (Kutuların altında kalsın)
    for obj in tracked_objs:
        tid, cid = obj["id"], obj["class_id"]
        draw_foot_trail(frame, list(foot_trails[tid]), C_FOOT if cid == 0 else C_BLUE)

    # Hedef Kutuları ve Metinleri
    for obj in tracked_objs:
        tid, cid = obj["id"], obj["class_id"]
        x1, y1, x2, y2 = obj["bbox"]

        dist = distance_estimate(obj["bbox_h"], cid)
        speed = compute_speed(list(center_history[tid]))
        is_locked = (tid == locked_id)
        is_breached = (tid in breached_ids)

        # Kutu Rengi
        color = C_RED if is_breached else (C_CYAN if is_locked else (C_BLUE if cid in (2, 7) else C_GREEN))

        # Ana kutu ve köşeler
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        for (px, py), (dx, dy) in [((x1,y1),(1,1)), ((x2,y1),(-1,1)), ((x1,y2),(1,-1)), ((x2,y2),(-1,-1))]:
            cv2.line(frame, (px, py), (px+dx*14, py), color, 2)
            cv2.line(frame, (px, py), (px, py+dy*14), color, 2)

        # Etiketler (İsim, Mesafe, Hız, Tehdit)
        lbl = obj["label"] + (f"  {dist}m" if dist else "") + (" [LOCKED]" if is_locked else "")
        (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.50, 1)
        lbl_y = max(y1, th + 14)
        cv2.rectangle(frame, (x1, lbl_y-th-10), (x1+tw+8, lbl_y), color, -1)
        cv2.putText(frame, lbl, (x1+4, lbl_y-4), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (0,0,0), 1, cv2.LINE_AA)

        lvl, lvl_color = threat_level(dist, speed)
        cv2.putText(frame, f"THREAT:{lvl}  SPD:{speed:.0f}px/f", (x1, y2+16), cv2.FONT_HERSHEY_SIMPLEX, 0.42, lvl_color, 1, cv2.LINE_AA)

    # İhlal anında güvenlik bölgesini kırmızı ve yarı saydam göster
    if breach_timer > 0:
        overlay = frame.copy()
        cv2.rectangle(overlay, (zx1, zy1), (zx2, zy2), C_RED, 2)
        cv2.putText(overlay, "SECURITY ZONE", (zx1+6, zy1+18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, C_RED, 1)
        cv2.addWeighted(overlay, 0.35, frame, 0.65, 0, frame)

    out.write(frame)

cap.release()
out.release()
print(f"✅ İŞLEM TAMAMLANDI! Çıktı: {OUTPUT_PATH}")
from google.colab import files
files.download(OUTPUT_PATH)

🛡️ AEGIS v2.0 (output_v2) ANA MOTOR BAŞLATILIYOR...
✅ İŞLEM TAMAMLANDI! Çıktı: /content/output_v1.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>